# Kalman Filtering and Smoothing by Elimination

The Kalman filter is a staple of robotics state estimation, usually taught as a set of matrix equations for prediction, innovation covariance, and Kalman gain, yet its underlying structure is far more intuitive when viewed as variable elimination on a factor graph. Writing each time step as a small factor graph replaces ad-hoc matrix algebra with a single graphical operation—eliminating one variable at a time into a Bayes net—that generalizes directly to fixed-lag smoothing and nonlinear estimation:

- **A Kalman filter is implemented in GTSAM as two elimination steps:** eliminate $x_{k-1}$ to form the predictive prior on $x_k$, then eliminate $x_k$ to fuse that prior with the new measurement $z_k$.
- **A fixed-lag smoother simply extends the number of states:** keep a chain Bayes net of $L + 1$ states instead of one, run the same two eliminations at the newest end, and drop the oldest leaf.
- **Nonlinearity is accommodated by linearizing factors once or several times:** once at the prediction (EKF), repeatedly on the current update (IEKF), or across the active window (nonlinear fixed-lag smoothing).

For a compact visual overview with interactive step sliders, open [KalmanFilteringAndSmoothing.html](https://borglab.github.io/gtsam/KalmanFilteringAndSmoothing.html) or download {download}`KalmanFilteringAndSmoothing.html` (generated from this notebook; interactive charts use Plotly CDN and require an internet connection).

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/linear/KalmanFilteringAndSmoothing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam==4.3.0 plotly
    !curl -sSLO https://raw.githubusercontent.com/borglab/gtsam/develop/python/gtsam/examples/linear/kalman_plots.py
except ImportError:
    pass

In [2]:
import gtsam
import numpy as np
from gtsam.symbol_shorthand import B, X

import kalman_plots as plots

## Linear filtering with 2D measurements

A 2D tracking problem with direct position measurements strips away algebraic clutter so the equivalence between the Kalman gain and factor-graph elimination is immediately visible. The robot's state is a 2D position $x_k \in \mathbb{R}^2$ (`gtsam.Point2`). At each step it moves by a known displacement $u$ with Gaussian process noise and receives a noisy 2D position measurement $z_k$:

$$
x_k = x_{k-1} + u + w_k,\quad w_k\sim\mathcal{N}(0,Q),
\qquad
z_k = x_k + v_k,\quad v_k\sim\mathcal{N}(0,R).
$$

The process and sensor noises are independent across time and from the initial state. We start with a Gaussian prior on $x_0$ and take the first measurement at $k=1$.

In [3]:
sample_count = 36
rng = np.random.default_rng(12)
u, m0, I = gtsam.Point2(0.18, 0.07), gtsam.Point2(0.0, 0.0), np.eye(2)
P0, Q, R = 0.60**2 * I, 0.09**2 * I, 0.18**2 * I

truth = np.empty((sample_count, 2))
truth[0] = rng.multivariate_normal(m0, P0)
for k in range(1, sample_count):
    truth[k] = truth[k - 1] + u + rng.multivariate_normal(np.zeros(2), Q)

measurements = truth + rng.multivariate_normal(np.zeros(2), R, sample_count)
measurements[0] = np.nan  # No measurement at time zero.

### The textbook Kalman filter

The standard Kalman filter updates a Gaussian belief $\mathcal{N}(m_{k-1}, P_{k-1})$ through two matrix formulas—one to propagate uncertainty forward and one to weight the new measurement by the Kalman gain:

1. **Predict** forward through the motion model:
   $$
   m_k^- = m_{k-1} + u, \qquad P_k^- = P_{k-1} + Q.
   $$
2. **Update** with the new 2D measurement $z_k$ using the Kalman gain $K_k$ (with $H = I$):
   $$
   K_k = P_k^- (P_k^- + R)^{-1}, \qquad
   m_k = m_k^- + K_k (z_k - m_k^-), \qquad
   P_k = (I - K_k) P_k^-.
   $$

The Kalman gain $K_k$ blends the prediction and the measurement according to their relative uncertainties. Next we show that this entire predict–update recursion is just two variable-elimination steps on a two-state factor graph.

In [4]:
mean, covariance = m0.copy(), P0.copy()
kf_mean, kf_covariance = [mean], [covariance]
kf_pred_mean, kf_pred_cov = [None], [None]
for k in range(1, sample_count):
    mean_pred, covariance_pred = mean + u, covariance + Q
    kf_pred_mean.append(mean_pred)
    kf_pred_cov.append(covariance_pred)
    gain = np.linalg.solve(covariance_pred + R, covariance_pred).T
    mean = mean_pred + gain @ (measurements[k] - mean_pred)
    covariance = (I - gain) @ covariance_pred
    kf_mean.append(mean)
    kf_covariance.append(covariance)

In [5]:
plots.plot_kf_ellipses(
    truth, measurements, kf_pred_mean, kf_pred_cov, kf_mean, kf_covariance
).show()

### Demystifying the Kalman filter by two-stage elimination

Viewed as a factor graph, the textbook predict–update cycle reduces to two sequential variable-elimination steps on a two-state fragment. Represent step $k$ with the previous state $x_{k-1}$, the current state $x_k$, the accumulated prior $p$ for $p(x_{k-1}\mid z_{1:k-1})$, the motion factor $u$, and the 2D measurement $z_k$:

![Two-state Kalman step factor graph](figures/KalmanStepFactorGraph.svg)

Eliminating this fragment in order $(x_{k-1}, x_k)$ proceeds in **two stages** and produces a two-node Bayes net:

1. **Stage 1 — Eliminate $x_{k-1}$ (Predict):** Eliminating $x_{k-1}$ extracts the conditional $p(x_{k-1} \mid x_k)$ and leaves **two unary factors on $x_k$**: the **predictive prior** $p^-(x_k) = p(x_k \mid z_{1:k-1})$ and the **measurement factor** $z_k$.
2. **Stage 2 — Eliminate $x_k$ (Update):** Eliminating $x_k$ fuses $p^-$ and $z_k$ into the single marginal posterior density $p(x_k \mid z_{1:k})$—the exact same Gaussian posterior computed by the Kalman gain above—after which the conditional $p(x_{k-1} \mid x_k)$ is dropped.

![Two-node Kalman step Bayes net](figures/KalmanStepBayesNet.svg)

In GTSAM, `eliminateSequential` performs both stages and returns the two-node Bayes net, while `eliminatePartialSequential([X(k - 1)])` performs Stage 1 and returns the remaining factor graph on $x_k$.

In [6]:
prior_noise = gtsam.noiseModel.Isotropic.Sigma(2, np.sqrt(P0[0, 0]))
motion_noise = gtsam.noiseModel.Isotropic.Sigma(2, np.sqrt(Q[0, 0]))
measurement_noise = gtsam.noiseModel.Isotropic.Sigma(2, np.sqrt(R[0, 0]))


def initial_graph():
    graph = gtsam.GaussianFactorGraph()
    graph.add(X(0), I, m0, prior_noise)
    return graph


def add_step(graph, k):
    graph.add(X(k - 1), -I, X(k), I, u, motion_noise)
    graph.add(X(k), I, measurements[k], measurement_noise)


def eliminate_oldest(graph, k):
    _, remaining = graph.eliminatePartialSequential(gtsam.Ordering([X(k)]))
    return remaining


# Eliminate step 1 in order [X(0), X(1)] to inspect the resulting two-node DAG.
step1 = initial_graph()
add_step(step1, 1)
dag = step1.eliminateSequential(gtsam.Ordering([X(0), X(1)]))
print("Two-node DAG for step 1:")
for i in range(dag.size()):
    names = [gtsam.DefaultKeyFormatter(k) for k in dag.at(i).keys()]
    rhs = f" | {', '.join(names[1:])}" if len(names) > 1 else ""
    print(f"  p({names[0]}{rhs})")

# Filter across all steps, recording predictive (after motion) and posterior (after measurement) densities.
pred_mean, pred_cov = [None], [None]
filtered, filter_cov = np.empty_like(truth), np.empty((sample_count, 2, 2))
filtered[0], filter_cov[0] = m0, P0
posterior = initial_graph()
for k in range(1, sample_count):
    posterior.add(X(k - 1), -I, X(k), I, u, motion_noise)
    posterior = eliminate_oldest(posterior, k - 1)
    pred_tree = posterior.eliminateMultifrontal()
    pred_mean.append(pred_tree.optimize().at(X(k)))
    pred_cov.append(pred_tree.marginalCovariance(X(k)))

    posterior.add(X(k), I, measurements[k], measurement_noise)
    tree = posterior.eliminateMultifrontal()
    filtered[k] = tree.optimize().at(X(k))
    filter_cov[k] = tree.marginalCovariance(X(k))

np.testing.assert_allclose(pred_mean[1:], kf_pred_mean[1:], rtol=0, atol=1e-10)
np.testing.assert_allclose(pred_cov[1:], kf_pred_cov[1:], rtol=0, atol=1e-10)
np.testing.assert_allclose(filtered, kf_mean, rtol=0, atol=1e-10)
np.testing.assert_allclose(filter_cov, kf_covariance, rtol=0, atol=1e-10)
print(f"Max absolute mean difference:       {np.max(np.abs(filtered - kf_mean)):.2e}")
print(f"Max absolute covariance difference: {np.max(np.abs(filter_cov - kf_covariance)):.2e}")

Two-node DAG for step 1:
  p(x0 | x1)
  p(x1)
Max absolute mean difference:       3.55e-15
Max absolute covariance difference: 6.59e-17


In [7]:
plots.plot_kf_vs_elimination(
    truth, measurements,
    kf_pred_mean, kf_pred_cov, kf_mean, kf_covariance,
    pred_mean, pred_cov, filtered, filter_cov,
).show()

## Linear fixed-lag smoothing

Once filtering is understood as variable elimination on a Bayes net, extending it to a fixed-lag smoother requires no new matrix derivations—only keeping a longer chain of states before dropping the oldest leaf. In steady state, a fixed-lag smoother holds a chain Bayes net on $L + 1 = 4$ states ($x_{k-4} \leftarrow x_{k-3} \leftarrow x_{k-2} \leftarrow x_{k-1}$) and never needs to re-eliminate older states:

1. **Liquify only the newest root:** Turn $p(x_{k-1})$ into a unary factor $p$ and attach the new motion factor $u$ and measurement $z_k$ for $x_k$—the exact same two-state fragment as the Kalman filter at the right end.
2. **Run the same two eliminations at the end:** Eliminate $x_{k-1}$ (extending the chain with $p(x_{k-1} \mid x_k)$ and leaving $p^-$ and $z_k$ on $x_k$), then eliminate $x_k$ to form the updated root $p(x_k \mid z_{1:k})$.
3. **Marginalize by dropping the oldest leaf:** Because $x_{k-4}$ is already a leaf conditional $p(x_{k-4} \mid x_{k-3})$, dropping it leaves the exact 4-node Bayes net on $x_{k-3} \dots x_k$.

Equivalently, we can maintain the 4-state factor graph window and eliminate the oldest variable when the window fills:

![Four-state linear fixed-lag factor graph](figures/FixedLagFactorGraph.svg)

Eliminating the active window yields the same four-node chain Bayes net:

![Four-state linear fixed-lag Bayes net](figures/FixedLagBayesNet.svg)

The root $p(x_k \mid z_{1:k})$ still matches the Kalman filter, while back-substitution propagates newer measurements backward from $x_k$ through $x_{k-3}$ to tighten the older marginals.

In [8]:
lag = 3  # Keep L + 1 = 4 states in the active window.
window = initial_graph()
lagged, lagged_cov = np.empty_like(truth), np.empty_like(filter_cov)
window_snapshots = {}
for k in range(sample_count):
    if k:
        add_step(window, k)
    first = max(0, k - lag)
    if k > lag:
        window = eliminate_oldest(window, first - 1)
    assert set(window.keyVector()) == {X(j) for j in range(first, k + 1)}

    tree = window.eliminateMultifrontal()
    mean = tree.optimize()
    np.testing.assert_allclose(mean.at(X(k)), filtered[k], rtol=0, atol=1e-10)
    if lag <= k <= 24:
        indices = list(range(first, k + 1))
        window_snapshots[k] = (
            indices,
            [mean.at(X(j)) for j in indices],
            [tree.marginalCovariance(X(j)) for j in indices],
        )

    # Publish x_j after lag extra measurements, or at the end of the stream if sooner.
    publish = range(first, k + 1) if k == sample_count - 1 else ([first] if k >= lag else [])
    for j in publish:
        lagged[j] = mean.at(X(j))
        lagged_cov[j] = tree.marginalCovariance(X(j))

assert np.all(np.trace(filter_cov - lagged_cov, axis1=1, axis2=2) >= -1e-10)
print(f"Steady-state RMS uncertainty: filter {np.sqrt(np.trace(filter_cov[10])):.3f} m "
      f"vs fixed-lag ({lag + 1} states) {np.sqrt(np.trace(lagged_cov[10])):.3f} m")

Steady-state RMS uncertainty: filter 0.159 m vs fixed-lag (4 states) 0.127 m


In [9]:
plots.plot_marginals_snake(
    truth, measurements, {k: window_snapshots[k] for k in (4, 5, 6)}, lagged, lagged_cov
).show()

The right panel below plots $\sqrt{\operatorname{tr}(P_k)}$, the posterior RMS position uncertainty under the model. Waiting just three extra steps cuts uncertainty noticeably across the trajectory, while at the final step both estimators have seen the same data and coincide.

In [10]:
plots.plot_linear_smoothing(
    truth, measurements, filtered, filter_cov, lagged, lagged_cov, lag
).show()

## A range update is only locally linear

Real robotics sensors such as range beacons are nonlinear, so the factor graph can no longer be solved in closed form without first linearizing its factors around an operating point. Here a robot drives at constant speed and turn rate around a roughly circular path with state $x=(p_x,p_y,\theta)$ (heading in unwrapped radians). The nominal motion adds $d(\cos a,\sin a)$ to position and $\omega\Delta t$ to heading, where $d=2(v/\omega)\sin(\omega\Delta t/2)$ and $a=\theta+\omega\Delta t/2$. Each beacon $b_i$ gives **one scalar range** $h_i(x)=\|p-b_i\|$, so the direction in which a range pulls the estimate depends on where we linearize it.

Both filters predict a Gaussian with mean $m^-=f(m)$ and covariance $P^-=FPF^\top+Q$, where $F$ is the motion Jacobian at the previous estimate. Given the two independent ranges $z$, the local MAP problem is

$$
\min_x \frac12\|x-m^-\|_{(P^-)^{-1}}^2
       +\frac12\|z-h(x)\|_{R^{-1}}^2.
$$

The **EKF** takes one Gauss–Newton step from $m^-$, linearizing $h$ there. The **iterated EKF (IEKF)** takes further steps on the **same** objective. At iterate $x_i$, with $H_i=Dh(x_i)$ and $K_i=P^-H_i^\top(H_i P^-H_i^\top+R)^{-1}$, the next iterate is

$$
x_{i+1}=m^-+K_i\left[z-h(x_i)+H_i(x_i-m^-)\right].
$$

The term $H_i(x_i-m^-)$ keeps the prior centered at $m^-$: we do not feed a posterior back as a prior or count the ranges again. We compute the covariance once, using the chosen linearization of the range factors. Iterating the correction does not redo the motion prediction or guarantee a better estimate.

In [11]:
rng_circle = np.random.default_rng(7)
beacons = np.array([[-3.0, 2.0], [7.0, 5.0]])
dt_circle, speed, yaw_rate = 0.35, 2.0, 0.42
turn = yaw_rate * dt_circle
step_length = speed * dt_circle * np.sinc(turn / (2 * np.pi))
Q_circle = np.diag([0.04**2, 0.04**2, 0.025**2])
range_sd = 0.18
R_circle = range_sd**2 * np.eye(len(beacons))
initial_pose = np.array([-2.5, -3.0, 0.5])
P_initial = np.diag([3.5**2, 3.5**2, 0.6**2])


def drive(x):
    angle = x[2] + turn / 2
    c, s = np.cos(angle), np.sin(angle)
    F = np.eye(3)
    F[:2, 2] = step_length * np.array([-s, c])
    return x + np.array([step_length * c, step_length * s, turn]), F


def beacon_ranges(x):
    delta = x[:2] - beacons
    h = np.linalg.norm(delta, axis=1)
    H = np.column_stack((delta / h[:, None], np.zeros(len(beacons))))
    return h, H


circle_truth, circle_measurements = [np.zeros(3)], []
for _ in range(48):
    pose = drive(circle_truth[-1])[0] + rng_circle.multivariate_normal(np.zeros(3), Q_circle)
    circle_truth.append(pose)
    circle_measurements.append(beacon_ranges(pose)[0] + rng_circle.normal(0, range_sd, len(beacons)))
circle_truth = np.array(circle_truth)

In [12]:
def range_update(mean, S, z, max_steps):
    x, trace = mean.copy(), [mean.copy()]
    for _ in range(max_steps):
        h, H = beacon_ranges(x)
        K = np.linalg.solve(H @ S @ H.T + R_circle, H @ S).T
        candidate = mean + K @ (z - h + H @ (x - mean))
        trace.append(candidate)
        if np.linalg.norm(candidate - x) < 1e-8:
            x = candidate
            break
        x = candidate
    if max_steps > 1:
        _, H = beacon_ranges(x)
        K = np.linalg.solve(H @ S @ H.T + R_circle, H @ S).T
    U = np.eye(3) - K @ H
    return x, U @ S @ U.T + K @ R_circle @ K.T, np.array(trace)


def follow_circle(max_steps):
    x, P = initial_pose.copy(), P_initial.copy()
    poses = [x]
    for z in circle_measurements:
        mean, F = drive(x)
        S = F @ P @ F.T + Q_circle
        x, P = range_update(mean, S, z, max_steps)[:2] if max_steps else (mean, S)
        poses.append(x)
    return np.array(poses)


circle_estimates = {
    "Dead reckoning": follow_circle(0),
    "EKF": follow_circle(1),
    "Iterated EKF": follow_circle(12),
}
print(f"{'Position RMSE (m)':<20} {'First 8 steps':>14} {'All steps':>12}")
for label, estimates in circle_estimates.items():
    sq_err = np.sum((estimates[1:, :2] - circle_truth[1:, :2]) ** 2, axis=1)
    print(f"{label:<20} {np.sqrt(sq_err[:8].mean()):14.3f} {np.sqrt(sq_err.mean()):12.3f}")

first_mean, first_F = drive(initial_pose)
first_P = first_F @ P_initial @ first_F.T + Q_circle
first_z = circle_measurements[0]
first_trace = range_update(first_mean, first_P, first_z, max_steps=12)[2]
np.testing.assert_allclose(first_trace[1], circle_estimates["EKF"][1])

def first_objective(x):
    dp, dr = x - first_mean, first_z - beacon_ranges(x)[0]
    return 0.5 * (dp @ np.linalg.solve(first_P, dp) + dr @ np.linalg.solve(R_circle, dr))

print(f"First update objective: prior {first_objective(first_trace[0]):.2f}, "
      f"EKF {first_objective(first_trace[1]):.2f}, IEKF {first_objective(first_trace[-1]):.2f}")

Position RMSE (m)     First 8 steps    All steps
Dead reckoning                3.876        6.059
EKF                           0.961        0.535
Iterated EKF                  0.355        0.228
First update objective: prior 231.29, EKF 38.96, IEKF 0.61


In [13]:
plots.plot_circle_ekf(
    circle_truth, circle_estimates, beacons, first_trace, first_objective
).show()

The startup error makes the EKF's first range linearization poor. Moving the linearization point lowers the very same first-step objective and, in this run, improves position RMSE most during startup. Once the estimates are closer to the truth, less relinearization is needed. Try changing the initial pose or the beacons: iterating an ambiguous or badly initialized fit does not guarantee finding the right answer.

In graph terms, both filters use a predicted Gaussian prior and the same current range factors. The IEKF relinearizes only those measurements against that fixed prediction. A nonlinear fixed-lag smoother instead keeps several states and can revise both recent motion and measurement linearizations before marginalizing the oldest states; it cannot relinearize nonlinear factors that have already been absorbed into a Gaussian boundary factor.

## Nonlinear fixed-lag smoothing in GTSAM

Combining a multi-state window with repeated linearization lets GTSAM's fixed-lag smoothers revise recent poses and nonlinear factors together while keeping computation bounded. On the circular problem above, keeping a 4-state window lets the smoother relinearize both recent motion factors (`BetweenFactorPose2`) and recent beacon ranges (`RangeFactor2D`) before marginalizing expired poses. Pinning the two known beacons $b_0, b_1$ inside the active window gives the four-state nonlinear factor graph below:

![Four-state nonlinear fixed-lag factor graph with known beacons](figures/NonlinearFixedLagFactorGraph.svg)

- [BatchFixedLagSmoother](https://github.com/borglab/gtsam/blob/develop/gtsam/nonlinear/doc/BatchFixedLagSmoother.ipynb) optimizes the active window with Levenberg–Marquardt, linearizing and partially eliminating expired keys into boundary [LinearContainerFactors](https://github.com/borglab/gtsam/blob/develop/gtsam/nonlinear/LinearContainerFactor.h).
- [IncrementalFixedLagSmoother](https://github.com/borglab/gtsam/blob/develop/gtsam/nonlinear/doc/IncrementalFixedLagSmoother.ipynb) updates an iSAM2 Bayes tree incrementally and marginalizes expired leaves via `ISAM2::marginalizeLeaves`.

Both take new factors, initial values for new keys, and timestamps. Here we run both with the same 4-state window (`lag = 3`) on the circular range-measurement trajectory above.

In [14]:
delta_pose = gtsam.Pose2(step_length * np.cos(turn / 2), step_length * np.sin(turn / 2), turn)
circle_prior_noise = gtsam.noiseModel.Diagonal.Sigmas([3.5, 3.5, 0.6])
circle_motion_noise = gtsam.noiseModel.Diagonal.Sigmas([0.04, 0.04, 0.025])
circle_range_noise = gtsam.noiseModel.Isotropic.Sigma(1, range_sd)
beacon_pin_noise = gtsam.noiseModel.Isotropic.Sigma(2, 1e-4)


def run_nonlinear_smoother(smoother):
    n_steps = len(circle_measurements)
    estimates = np.empty((n_steps + 1, 2))
    for k in range(n_steps + 1):
        factors, values = gtsam.NonlinearFactorGraph(), gtsam.Values()
        timestamps = {X(k): float(k), B(0): float(k), B(1): float(k)}
        if k == 0:
            p0 = gtsam.Pose2(*initial_pose)
            factors.add(gtsam.PriorFactorPose2(X(0), p0, circle_prior_noise))
            values.insert(X(0), p0)
            for i, b in enumerate(beacons):
                factors.add(gtsam.PriorFactorPoint2(B(i), gtsam.Point2(*b), beacon_pin_noise))
                values.insert(B(i), gtsam.Point2(*b))
        else:
            factors.add(gtsam.BetweenFactorPose2(X(k - 1), X(k), delta_pose, circle_motion_noise))
            for i, z in enumerate(circle_measurements[k - 1]):
                factors.add(gtsam.RangeFactor2D(X(k), B(i), float(z), circle_range_noise))
            prev = smoother.calculateEstimatePose2(X(k - 1))
            values.insert(X(k), prev.compose(delta_pose))

        smoother.update(factors, values, timestamps)
        est = smoother.calculateEstimate()
        first = max(0, k - lag)
        publish = range(first, k + 1) if k == n_steps else ([first] if k >= lag else [])
        for j in publish:
            p = est.atPose2(X(j))
            estimates[j] = [p.x(), p.y()]
    return estimates


nonlinear_smoothers = {
    f"Batch fixed-lag ({lag + 1} states)": run_nonlinear_smoother(gtsam.BatchFixedLagSmoother(float(lag))),
    f"Incremental fixed-lag ({lag + 1} states)": run_nonlinear_smoother(gtsam.IncrementalFixedLagSmoother(float(lag))),
}
print(f"{'Position RMSE (m)':<34} {'First 8 steps':>14} {'All steps':>12}")
for label, pos in (
    ("EKF", circle_estimates["EKF"][:, :2]),
    ("Iterated EKF", circle_estimates["Iterated EKF"][:, :2]),
    *nonlinear_smoothers.items(),
):
    sq_err = np.sum((pos[1:] - circle_truth[1:, :2]) ** 2, axis=1)
    print(f"{label:<34} {np.sqrt(sq_err[:8].mean()):14.3f} {np.sqrt(sq_err.mean()):12.3f}")

Position RMSE (m)                   First 8 steps    All steps
EKF                                         0.961        0.535
Iterated EKF                                0.355        0.228
Batch fixed-lag (4 states)                  0.215        0.121
Incremental fixed-lag (4 states)            0.514        0.233


In [15]:
plots.plot_nonlinear_smoothing(
    circle_truth,
    circle_estimates["EKF"][:, :2],
    circle_estimates["Iterated EKF"][:, :2],
    nonlinear_smoothers,
    beacons,
).show()

### What changes across estimators?

| Estimator | States kept after the update | What happens to old information |
| --- | --- | --- |
| Kalman filter | Current state only | Exactly summarized by one Gaussian posterior |
| EKF / iterated EKF | Current state only | Approximately Gaussian; iteration refines the current nonlinear update |
| Fixed-lag smoother | Recent window ($L + 1$ states) | Summarized at the window boundary; active states and nonlinear factors in the window can still be revised |

A larger window changes **which states** you can revise. Relinearization changes **how accurately** you approximate the nonlinear problem at the states you keep. These are separate choices.

In [16]:
import kalman_html


def _rmse(pos):
    sq_err = np.sum((pos[1:] - circle_truth[1:, :2]) ** 2, axis=1)
    return float(np.sqrt(sq_err[:8].mean())), float(np.sqrt(sq_err.mean()))


batch_pos = nonlinear_smoothers[f"Batch fixed-lag ({lag + 1} states)"]
inc_pos = nonlinear_smoothers[f"Incremental fixed-lag ({lag + 1} states)"]
rmse_rows = []
for label, pos in (
    ("Dead reckoning", circle_estimates["Dead reckoning"][:, :2]),
    ("EKF", circle_estimates["EKF"][:, :2]),
    ("Iterated EKF", circle_estimates["Iterated EKF"][:, :2]),
    (f"Batch fixed-lag ({lag + 1} states)", batch_pos),
    (f"Incremental fixed-lag ({lag + 1} states)", inc_pos),
):
    first8, all_steps = _rmse(pos)
    rmse_rows.append({"label": label, "first8": first8, "all": all_steps})

payload = {
    "linear": {
        "u": u.tolist(),
        "m0": m0.tolist(),
        "P0": P0.tolist(),
        "sigma_q": float(np.sqrt(Q[0, 0])),
        "sigma_r": float(np.sqrt(R[0, 0])),
        "truth": truth.tolist(),
        "measurements": [[None, None] if k == 0 else measurements[k].tolist() for k in range(sample_count)],
        "filter_cov": filter_cov.tolist(),
        "max_mean_diff": float(np.max(np.abs(filtered - kf_mean))),
    },
    "fls": {
        "lag": int(lag),
        "lagged": lagged.tolist(),
        "lagged_cov": lagged_cov.tolist(),
        "snapshots": {
            str(k): {
                "indices": indices,
                "means": [m.tolist() for m in means],
                "covs": [c.tolist() for c in covs],
            }
            for k, (indices, means, covs) in window_snapshots.items()
        },
    },
    "nonlinear": {
        "beacons": beacons.tolist(),
        "truth": circle_truth[:, :2].tolist(),
        "dead_reckoning": circle_estimates["Dead reckoning"][:, :2].tolist(),
        "ekf": circle_estimates["EKF"][:, :2].tolist(),
        "iekf": circle_estimates["Iterated EKF"][:, :2].tolist(),
        "batch_fls": batch_pos.tolist(),
        "inc_fls": inc_pos.tolist(),
        "first_z": first_z.tolist(),
        "first_trace": first_trace[:, :2].tolist(),
        "first_objectives": [float(first_objective(x)) for x in first_trace],
        "rmse_rows": rmse_rows,
    },
}
kalman_html.export_html("KalmanFilteringAndSmoothing.html", payload)
print("Exported KalmanFilteringAndSmoothing.html with embedded notebook payload.")

Exported KalmanFilteringAndSmoothing.html with embedded notebook payload.
